# WPP whole-video decision explorer

This UI reads a checkpoint; it never retrains or changes datasets/checkpoints. Open and compute are explicit buttons because attribution can be costly. See `docs/wpp-probing.md` for browser-media setup and limitations.

In [1]:
# Locate the checkout independently of the notebook's current working directory.
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in (here, *here.parents):
    if (candidate / 'src').is_dir() and (candidate / 'README.md').is_file():
        REPO_ROOT = candidate
        break
else:
    raise RuntimeError('Could not find the project root containing src/. Start Jupyter in or below the checkout.')
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f'Using project: {REPO_ROOT}')

Using project: /Users/naoromi/projects/project entropy/JEPA_Entropy/Proj-Entropy/WPP_Classification_Model


In [2]:
# ipywidgets is the only optional notebook dependency (requirements-notebook.txt).
try:
    import ipywidgets as widgets
except ImportError as exc:
    raise RuntimeError('Install the optional UI controls: python -m pip install -r requirements-notebook.txt') from exc
from IPython.display import display
import inspect
import json

from src.wpp_explain import open_explainer, list_videos, explain_video, export_explanation
from src.wpp_explain_types import DataRoots
from src.wpp_notebook_ui import ExplorerState, jupyter_files_url, render_explanation_iframe


## Configure and open

`DataRoots JSON` is passed as keyword arguments to `DataRoots`; use the field names shown by the backend type. A `/files` prefix is valid only for media physically under the Jupyter server root. Otherwise use a browser-reachable per-video URL override after selecting a sample.

In [3]:
roots_json = widgets.Textarea(
    description='DataRoots JSON', layout=widgets.Layout(width='720px', height='90px'),
    placeholder='{\"packed_bundle\": \"/path/to/bundle\", \"source_manifest\": \"/path/source-manifest-directory\", \"index\": \"/path/extraction-index-directory\", \"source_video_root\": \"/path/media\"}')
run_dir = widgets.Text(description='Run directory', layout=widgets.Layout(width='620px'))
checkpoint = widgets.Text(value='best', description='Checkpoint')
device = widgets.Text(value='auto', description='Device (auto/cuda:0/mps/cpu)')
partition = widgets.Dropdown(options=['test', 'validation', 'train'], value='test', description='Partition')
open_button = widgets.Button(description='Open', button_style='primary')
video_select = widgets.Dropdown(options=[], description='Video', layout=widgets.Layout(width='720px'))
ig_steps = widgets.BoundedIntText(value=32, min=2, max=2048, description='IG steps')
attention_chunk = widgets.IntText(value=0, description='Attention chunk (0=auto)')
display_mode = widgets.Dropdown(options=['positive', 'signed'], value='positive', description='Initial IG mode')
compute_button = widgets.Button(description='Compute explanation', button_style='warning', disabled=True)
files_prefix = widgets.Text(description='/files prefix', placeholder='originals (only below Jupyter server root)', layout=widgets.Layout(width='620px'))
video_url_override = widgets.Text(description='Video URL override', placeholder='https://…/video.mp4', layout=widgets.Layout(width='720px'))
show_button = widgets.Button(description='Show viewer', disabled=True)
export_dir = widgets.Text(description='Fresh export dir', layout=widgets.Layout(width='720px'))
export_button = widgets.Button(description='Export figures', disabled=True)
status = widgets.Output(layout={'border': '1px solid #ddd'})
viewer_output = widgets.Output()
display(widgets.VBox([roots_json, run_dir, widgets.HBox([checkpoint, device, partition, open_button]), video_select, widgets.HBox([ig_steps, attention_chunk, display_mode, compute_button]), files_prefix, video_url_override, widgets.HBox([show_button, export_dir, export_button]), status, viewer_output]))

state = ExplorerState()
def report(event):
    # Backend progress is a ProgressEvent; accept a plain string as well.
    text = getattr(event, 'message', event)
    stage = getattr(event, 'stage', None)
    with status:
        print(f'[{stage}] {text}' if stage else text)
def clear_status():
    status.clear_output(wait=True)
def selected_id():
    return video_select.value
def video_field(video, name, default=None):
    return video.get(name, default) if isinstance(video, dict) else getattr(video, name, default)
def invalidate_explanation():
    state.invalidate_explanation()
    viewer_output.clear_output(wait=True)
    show_button.disabled = export_button.disabled = True

def media_url(explanation_video):
    override = video_url_override.value.strip()
    if override:
        return override
    backend_url = getattr(getattr(state.explanation, 'media', None), 'media_url', None)
    if backend_url:
        return backend_url
    prefix = files_prefix.value.strip()
    relative = video_field(explanation_video, 'source_relative_path')
    if prefix and relative:
        return jupyter_files_url(prefix, str(relative))
    return None

def refresh_partition():
    invalidate_explanation()
    state.videos = {}
    video_select.options = []
    compute_button.disabled = True
    if state.session is None:
        return 0
    videos = list_videos(state.session, partition.value)
    state.videos = {str(video_field(v, 'video_id')): v for v in videos}
    video_select.options = [(str(video_field(v, 'video_id')), str(video_field(v, 'video_id'))) for v in videos]
    compute_button.disabled = not bool(videos)
    return len(videos)


In [4]:
def on_open(_):
    clear_status()
    invalidate_explanation()
    state.session = None
    state.videos = {}
    video_select.options = []
    compute_button.disabled = True
    try:
        roots = DataRoots(**json.loads(roots_json.value))
        kwargs = {'checkpoint': checkpoint.value.strip() or 'best', 'roots': roots, 'device': device.value.strip() or 'auto', 'progress': report}
        if run_dir.value.strip():
            kwargs['run_dir'] = run_dir.value.strip()
        state.session = open_explainer(**kwargs)
        count = refresh_partition()
        report(f'Opened {count} {partition.value} videos. Select one, then compute.')
    except Exception as exc:
        state.session = None
        state.videos = {}
        invalidate_explanation()
        compute_button.disabled = True
        report(f'Open failed: {type(exc).__name__}: {exc}')

def on_partition_change(change):
    if change['name'] != 'value':
        return
    clear_status()
    try:
        count = refresh_partition()
        if state.session is not None:
            report(f'Loaded {count} {partition.value} videos. Select one, then compute.')
    except Exception as exc:
        state.videos = {}
        video_select.options = []
        invalidate_explanation()
        compute_button.disabled = True
        report(f'Partition refresh failed: {type(exc).__name__}: {exc}')

def on_video_change(change):
    if change['name'] == 'value':
        invalidate_explanation()

def on_compute(_):
    clear_status()
    invalidate_explanation()
    try:
        video_id = selected_id()
        if state.session is None or not video_id or video_id not in state.videos:
            raise RuntimeError('Open a partition and select a video first.')
        chunk = attention_chunk.value or None
        explanation = explain_video(state.session, partition.value, video_id, ig_steps=ig_steps.value, display_mode=display_mode.value, attention_chunk_queries=chunk, progress=report)
        explained_video = getattr(explanation, 'video', None)
        if video_field(explained_video, 'video_id') != video_id or video_field(explained_video, 'partition') != partition.value:
            raise RuntimeError('Backend returned an explanation for a different video or partition.')
        state.set_explanation(explanation, video_id=video_id, partition=partition.value)
        show_button.disabled = export_button.disabled = False
        report('Explanation computed once. Viewer interactions are now browser-only.')
    except Exception as exc:
        invalidate_explanation()
        report(f'Explanation failed: {type(exc).__name__}: {exc}')

def current_explanation():
    if not state.matches_selection(video_id=selected_id(), partition=partition.value):
        raise RuntimeError('Compute a fresh explanation for the current selected video and partition.')
    return state.explanation

def on_show(_):
    clear_status()
    try:
        explanation = current_explanation()
        url = media_url(getattr(explanation, 'video', None))
        if not url:
            raise RuntimeError('Set Video URL override, or /files prefix for media below the Jupyter server root. Never use file://.')
        with viewer_output:
            viewer_output.clear_output(wait=True)
            display(widgets.HTML(value=render_explanation_iframe(explanation, video_url=url)))
    except Exception as exc:
        report(f'Viewer unavailable: {type(exc).__name__}: {exc}')

def on_export(_):
    clear_status()
    try:
        explanation = current_explanation()
        destination = Path(export_dir.value).expanduser()
        if destination.exists():
            raise RuntimeError('Export directory must be a new path; the backend never overwrites exports.')
        export_explanation(explanation, destination, write_png=True, write_svg=True)
        report(f'Exported figures and metadata to {destination}')
    except Exception as exc:
        report(f'Export failed: {type(exc).__name__}: {exc}')

open_button.on_click(on_open)
partition.observe(on_partition_change, names='value')
video_select.observe(on_video_change, names='value')
compute_button.on_click(on_compute)
show_button.on_click(on_show)
export_button.on_click(on_export)


### Reading the player

The player uses per-frame PTS and preceding-frame lookup, not an assumed FPS. It stops attribution at the model-covered endpoint (at most 3,000 valid frames), even if source playback is longer. Positive/signed IG and attention have separate fixed per-video scales. Attention is patch-resolution and not class-specific; IG is relative to the frozen train-mean baseline. Browser codec errors are actionable: use a reachable URL or explicitly make a separate VFR-preserving conversion—never silently convert originals.